In [7]:
import json
import os
from pathlib import Path
from tqdm import tqdm
import numpy as np
from openai import OpenAI
from sentence_transformers import SentenceTransformer, util
from scipy.special import rel_entr

# Metric Definitions

## Seg single & multi

In [8]:
def compute_kl_divergence(P_dict: dict, Q_dict: dict, epsilon=1e-9):
    keys = list(P_dict.keys())
    P = np.array([P_dict[k] for k in keys], dtype=float) + epsilon
    Q = np.array([Q_dict[k] for k in keys], dtype=float) + epsilon
    P = P / np.sum(P)
    Q = Q / np.sum(Q)
    kl_div = np.sum(rel_entr(P, Q))
    return float(kl_div)



def evaluate_metrics_from_logs(save_path):
    import json
    from pathlib import Path
    import numpy as np
    
    all_logs = json.loads(Path(save_path).read_text())
    N = len(all_logs)
    if N == 0: return {}
    
    trial_revenues = []
    trial_sws = []
    trial_rels = []
    trial_kls = []
    
    for logs in all_logs:
        rev = sum(log['payment'] for log in logs) / 3  
        sw = sum(log['social_welfare'] for log in logs)
        rel = sum(log['relevance'] for log in logs)
        
        kl_sum = 0.0
        round_count = 0
        for log in logs:
            if 'allocation' in log and 'q_tilde' in log:
                kl_sum += compute_kl_divergence(log['allocation'], log['q_tilde'])
                round_count += 1
                
        trial_revenues.append(rev)
        trial_sws.append(sw)
        trial_rels.append(rel)
        if round_count > 0:
            trial_kls.append(kl_sum)
        else:
            trial_kls.append(0.0)
            
    mean_rev = np.mean(trial_revenues)
    mean_sw = np.mean(trial_sws)
    mean_rel = np.mean(trial_rels)
    mean_kl = np.mean(trial_kls)
    
    se_rev = np.std(trial_revenues, ddof=1) / np.sqrt(N) if N > 1 else 0.0
    se_sw = np.std(trial_sws, ddof=1) / np.sqrt(N) if N > 1 else 0.0
    se_rel = np.std(trial_rels, ddof=1) / np.sqrt(N) if N > 1 else 0.0
    se_kl = np.std(trial_kls, ddof=1) / np.sqrt(N) if N > 1 else 0.0
    
    return {
        'Revenue per Ad': {'mean': mean_rev, 'se': se_rev},
        'Social Welfare': {'mean': mean_sw, 'se': se_sw},
        'Relevance': {'mean': mean_rel, 'se': se_rel},
        'KL Divergence': {'mean': mean_kl, 'se': se_kl}
    }



## QP single

In [9]:
def compute_metrics_from_file(
    save_path='trial_logs.json',
    bids_dict=None,
):
    all_logs = json.loads(Path(save_path).read_text())
    N = len(all_logs)

    all_metrics = []

    for logs in all_logs:
        revenue        = 0.0
        social_welfare = 0.0
        relevance      = 0.0
        kl_total       = 0.0

        for log in logs:
            winner      = log['winner']
            x_dict      = log['allocation']
            qt_dict     = log['q_tilde']          
            payments    = log['payments_dict']    
            winner_x    = log['winner_x']
            winner_q_raw = log['winner_q_raw']     
            f_tilde_q0  = log['f_tilde_q0']       
            q0_raw      = log['q0']             
            q_full_sum = log["q_full_sum"]

            # ── KL divergence ─────────────────────────────────────────
            names  = list(x_dict.keys())
            x_vec  = np.array([x_dict[n]  for n in names])
            qt_vec = np.array([qt_dict[n] for n in names])
            x_clip  = np.clip(x_vec,  1e-300, None)
            qt_clip = np.clip(qt_vec, 1e-300, None)
            kl_total += float(np.sum(x_clip * np.log(x_clip / qt_clip)))

            # ── Revenue ───────────────────────────────────────────────
            if winner != 'organic' and winner in payments and winner_x > 1e-12:
                p_tilde = payments[winner]
                p_imp   = p_tilde / winner_x 
                revenue += p_imp * winner_q_raw 

            # ── Social Welfare ────────────────────────────────────────
            if winner == 'organic':
                social_welfare += f_tilde_q0*q_full_sum
            else:
                v_i = bids_dict[winner]   
                social_welfare += v_i * qt_dict[winner]*q_full_sum

            # ── Relevance ─────────────────────────────────────────────
            if winner == 'organic':
                relevance += q0_raw
            else:
                relevance += winner_q_raw
        n_ads = sum(1 for log in logs if log['ad_injected'])  

        all_metrics.append({
            'Revenue per Ad':       revenue / n_ads if n_ads > 0 else 0.0, 
            'SocialWelfare': social_welfare,
            'Relevance':     relevance,
            'KLDivergence':  kl_total,
            'NumAds':        n_ads,
        })

    # ── Summary ───────────────────────────────────────────────────────
    keys = ['Revenue per Ad', 'SocialWelfare', 'Relevance', 'KLDivergence', 'NumAds']
    print(f"  {'Metric':20s}  {'Mean':>10}  {'SE':>10}")
    print(f"  {'-'*44}")
    results = {}
    for k in keys:
        vals = [m[k] for m in all_metrics]
        mu = np.mean(vals)
        std = np.std(vals)   
        se  = std / np.sqrt(len(vals))
        results[k] = {'mean': mu, 'se': se}
        print(f"  {k:20s}  {mu:>10.2f}   {se:>10.3f}")
    
    return results

## QP multi

In [10]:
def compute_metrics_vcg(save_path='QP_multi_trial_logs.json', bids_dict=None):
    all_logs = json.loads(Path(save_path).read_text())
    N        = len(all_logs)
    all_metrics = []

    for log in all_logs:
        ad_injected    = log['ad_injected']
        q0             = log['q0']
        q_raw          = log['q_raw']
        winning_ad_idx = log['winning_ad_idx']
        q_A_star_i     = {int(k): v for k, v in log['q_A_star_i'].items()}
        vcg_payments   = {int(k): v for k, v in log['vcg_payments'].items()}
        best_sw        = log['best_sw']


        social_welfare = best_sw

        relevance = sum(q_A_star_i.values())

        # NumAds: number of ads in A* (excluding organic)
        num_ads = len(winning_ad_idx)
        
        # Revenue per Ad
        revenue = sum(
            vcg_payments[i] * q_A_star_i[i]
            for i in winning_ad_idx
            if i in vcg_payments and i in q_A_star_i
        ) / num_ads if num_ads > 0 else 0.0

        all_metrics.append({
            'Revenue per Ad':      revenue,
            'SocialWelfare': social_welfare,
            'Relevance':     relevance,
            'NumAds':        num_ads,
        })

    keys = ['Revenue per Ad', 'SocialWelfare', 'Relevance', 'NumAds']
    print(f"  {'Metric':20s}  {'Mean':>10}  {'SE':>10}")
    print(f"  {'-' * 44}")

    results = {}
    for k in keys:
        vals = [m[k] for m in all_metrics]
        mean = float(np.mean(vals))
        se   = float(np.std(vals) / np.sqrt(N))
        results[k] = {'mean': mean, 'se': se, 'all': vals}
        print(f'  {k:20s}  {mean:>10.2f}  {se:>10.3f}')

    return results

# **Metrics Calculation**

## Senario 1

### $q_0^{(1)}$

In [11]:
with open('QP single/QP_single_with_replacement_logs_scenario_1.json') as f:
    data = json.load(f)

q0_values = []
for trial in data:
    for r in trial:
        if r['round'] == 1:
            q0_values.append(r['q0'])

q0 = np.array(q0_values)
mean = np.mean(q0)
se = np.std(q0, ddof=1) / np.sqrt(len(q0))
print(f'N = {len(q0)}')
print(f'Mean q0 (round 1) = {mean:.2f}')
print(f'SE = {se:.3f}')

N = 100
Mean q0 (round 1) = 0.80
SE = 0.001


### QP single & multi

In [13]:
bids_dict = {"SunWing Airlines": 3, "TropicStay": 3, "WanderBite": 2, "NovaSkin": 2, "GridPower Bank": 1}


print("\n\n" + "="*50)
print(f"--- 1. QP w/ rep.  (N={len(json.loads(Path('QP single/QP_single_with_replacement_logs_scenario_1.json').read_text()))} trials) ---")
results = compute_metrics_from_file(
    save_path = 'QP single/QP_single_with_replacement_logs_scenario_1.json',
    bids_dict = bids_dict,
)

print(f"\n--- 2. QP w/o repl.   (N={len(json.loads(Path('QP single/QP_single_without_replacement_logs_scenario_1.json').read_text()))} trials) ---")
results = compute_metrics_from_file(
    save_path = 'QP single/QP_single_without_replacement_logs_scenario_1.json',
    bids_dict = bids_dict,
)

print(f"\n--- 3. QP multi-alloc  (N={len(json.loads(Path('QP multi/QP_multi_logs_scenario_1.json').read_text()))} trials) ---")
results = compute_metrics_vcg(
    save_path = 'QP multi/QP_multi_logs_scenario_1.json',
    bids_dict = bids_dict,
)
print("="*50)



--- 1. QP w/ rep.  (N=100 trials) ---
  Metric                      Mean          SE
  --------------------------------------------
  Revenue per Ad              1.64        0.024
  SocialWelfare               5.90        0.056
  Relevance                   2.17        0.011
  KLDivergence                0.02        0.001
  NumAds                      2.15        0.079

--- 2. QP w/o repl.   (N=100 trials) ---
  Metric                      Mean          SE
  --------------------------------------------
  Revenue per Ad              1.63        0.024
  SocialWelfare               5.42        0.016
  Relevance                   2.12        0.010
  KLDivergence                0.01        0.001
  NumAds                      1.67        0.051

--- 3. QP multi-alloc  (N=100 trials) ---
  Metric                      Mean          SE
  --------------------------------------------
  Revenue per Ad              1.07       0.002
  SocialWelfare               5.99       0.004
  Relevance        

### Seg single & multi

In [14]:
print("\n\n" + "="*50)
print(f"--- 1. Seg w/ rep.  (N={len(json.loads(Path('Seg single & multi/Seg_single_with_replacement_logs_scenario_1.json').read_text()))} trials) ---")
results = evaluate_metrics_from_logs('Seg single & multi/Seg_single_with_replacement_logs_scenario_1.json')
for metric, stats in results.items():
    print(f"  {metric}: {stats['mean']:.2f} ± {stats['se']:.3f}")

print(f"\n--- 2. Seg w/o rep.  (N={len(json.loads(Path('Seg single & multi/Seg_single_without_replacement_logs_scenario_1.json').read_text()))} trials) ---")
results = evaluate_metrics_from_logs('Seg single & multi/Seg_single_without_replacement_logs_scenario_1.json')
for metric, stats in results.items():
    print(f"  {metric}: {stats['mean']:.2f} ± {stats['se']:.3f}")

print(f"\n--- 3. Seg multi-alloc (N={len(json.loads(Path('Seg single & multi/Seg_multi_logs_scenario_1.json').read_text()))} trials) ---")
results = evaluate_metrics_from_logs('Seg single & multi/Seg_multi_logs_scenario_1.json')
for metric, stats in list(results.items())[:-1]:
    print(f"  {metric}: {stats['mean']:.2f} ± {stats['se']:.3f}")
print("="*50)



--- 1. Seg w/ rep.  (N=100 trials) ---
  Revenue per Ad: 1.13 ± 0.049
  Social Welfare: 5.00 ± 0.112
  Relevance: 1.97 ± 0.017
  KL Divergence: 0.18 ± 0.001

--- 2. Seg w/o rep.  (N=100 trials) ---
  Revenue per Ad: 1.06 ± 0.041
  Social Welfare: 4.48 ± 0.062
  Relevance: 1.84 ± 0.011
  KL Divergence: 0.19 ± 0.004

--- 3. Seg multi-alloc (N=100 trials) ---
  Revenue per Ad: 0.88 ± 0.038
  Social Welfare: 4.46 ± 0.062
  Relevance: 1.82 ± 0.007


## Senario 2

### $q_0^{(1)}$

In [15]:
with open('QP single/QP_single_with_replacement_logs_scenario_2.json') as f:
    data = json.load(f)

q0_values = []
for trial in data:
    for r in trial:
        if r['round'] == 1:
            q0_values.append(r['q0'])

q0 = np.array(q0_values)
mean = np.mean(q0)
se = np.std(q0, ddof=1) / np.sqrt(len(q0))
print(f'N = {len(q0)}')
print(f'Mean q0 (round 1) = {mean:.2f}')
print(f'SE = {se:.3f}')

N = 100
Mean q0 (round 1) = 0.78
SE = 0.003


### QP single & multi

In [16]:
bids_dict = {"Velora": 3, "BookHaven": 3, "MassMart": 2, "EspressoEdge": 2}

print("\n\n" + "="*50)
print(f"--- 1. QP w/ rep.  (N={len(json.loads(Path('QP single/QP_single_with_replacement_logs_scenario_2.json').read_text()))} trials) ---")
results = compute_metrics_from_file(
    save_path = 'QP single/QP_single_with_replacement_logs_scenario_2.json',
    bids_dict = bids_dict,
)

print(f"\n--- 2. QP w/o repl.   (N={len(json.loads(Path('QP single/QP_single_without_replacement_logs_scenario_2.json').read_text()))} trials) ---")
results = compute_metrics_from_file(
    save_path = 'QP single/QP_single_without_replacement_logs_scenario_2.json',
    bids_dict = bids_dict,
)

print(f"\n--- 3. QP multi-alloc  (N={len(json.loads(Path('QP multi/QP_multi_logs_scenario_2.json').read_text()))} trials) ---")
results = compute_metrics_vcg(
    save_path = 'QP multi/QP_multi_logs_scenario_2.json',
    bids_dict = bids_dict,
)
print("="*50)



--- 1. QP w/ rep.  (N=100 trials) ---
  Metric                      Mean          SE
  --------------------------------------------
  Revenue per Ad              1.58        0.028
  SocialWelfare               5.80        0.057
  Relevance                   2.15        0.012
  KLDivergence                0.04        0.001
  NumAds                      2.03        0.085

--- 2. QP w/o repl.   (N=100 trials) ---
  Metric                      Mean          SE
  --------------------------------------------
  Revenue per Ad              1.53        0.042
  SocialWelfare               5.23        0.015
  Relevance                   2.11        0.014
  KLDivergence                0.02        0.000
  NumAds                      1.36        0.061

--- 3. QP multi-alloc  (N=100 trials) ---
  Metric                      Mean          SE
  --------------------------------------------
  Revenue per Ad              1.01       0.001
  SocialWelfare               5.82       0.004
  Relevance        

### Seg single & multi

In [25]:
print("\n\n" + "="*50)
print(f"--- 1. Seg w/ rep.  (N={len(json.loads(Path('Seg single & multi/Seg_single_with_replacement_logs_scenario_2.json').read_text()))} trials) ---")
results = evaluate_metrics_from_logs('Seg single & multi/Seg_single_with_replacement_logs_scenario_2.json')
for metric, stats in results.items():
    print(f"  {metric}: {stats['mean']:.2f} ± {stats['se']:.3f}")

print(f"\n--- 2. Seg w/o rep.  (N={len(json.loads(Path('Seg single & multi/Seg_single_without_replacement_logs_scenario_2.json').read_text()))} trials) ---")
results = evaluate_metrics_from_logs('Seg single & multi/Seg_single_without_replacement_logs_scenario_2.json')
for metric, stats in results.items():
    print(f"  {metric}: {stats['mean']:.2f} ± {stats['se']:.3f}")

print(f"\n--- 3. Seg multi-alloc (N={len(json.loads(Path('Seg single & multi/Seg_multi_logs_scenario_2.json').read_text()))} trials) ---")
results = evaluate_metrics_from_logs('Seg single & multi/Seg_multi_logs_scenario_2.json')
for metric, stats in list(results.items())[:-1]:
    print(f"  {metric}: {stats['mean']:.2f} ± {stats['se']:.3f}")
print("="*50)



--- 1. Seg w/ rep.  (N=100 trials) ---
  Revenue per Ad: 1.15 ± 0.045
  Social Welfare: 5.01 ± 0.087
  Relevance: 1.86 ± 0.017
  KL Divergence: 0.06 ± 0.000

--- 2. Seg w/o rep.  (N=100 trials) ---
  Revenue per Ad: 1.07 ± 0.041
  Social Welfare: 4.62 ± 0.037
  Relevance: 1.75 ± 0.007
  KL Divergence: 0.05 ± 0.001

--- 3. Seg multi-alloc (N=100 trials) ---
  Revenue per Ad: 0.82 ± 0.034
  Social Welfare: 4.47 ± 0.040
  Relevance: 1.70 ± 0.007


## Senario 3

### $q_0^{(1)}$

In [26]:
with open('QP single/QP_single_with_replacement_logs_scenario_3.json') as f:
    data = json.load(f)

q0_values = []
for trial in data:
    for r in trial:
        if r['round'] == 1:
            q0_values.append(r['q0'])

q0 = np.array(q0_values)
mean = np.mean(q0)
se = np.std(q0, ddof=1) / np.sqrt(len(q0))
print(f'N = {len(q0)}')
print(f'Mean q0 (round 1) = {mean:.2f}')
print(f'SE = {se:.3f}')

N = 100
Mean q0 (round 1) = 0.79
SE = 0.002


### QP single & multi

In [28]:
bids_dict = {"Velora": 2, "BookHaven": 1, "MassMart": 3, "EspressoEdge": 3}

print("\n\n" + "="*50)
print(f"--- 1. QP w/ rep.  (N={len(json.loads(Path('QP single/QP_single_with_replacement_logs_scenario_3.json').read_text()))} trials) ---")
results = compute_metrics_from_file(
    save_path = 'QP single/QP_single_with_replacement_logs_scenario_3.json',
    bids_dict = bids_dict,
)

print(f"\n--- 2. QP w/o repl.   (N={len(json.loads(Path('QP single/QP_single_without_replacement_logs_scenario_3.json').read_text()))} trials) ---")
results = compute_metrics_from_file(
    save_path = 'QP single/QP_single_without_replacement_logs_scenario_3.json',
    bids_dict = bids_dict,
)

print(f"\n--- 3. QP multi-alloc  (N={len(json.loads(Path('QP multi/QP_multi_logs_scenario_3.json').read_text()))} trials) ---")
results = compute_metrics_vcg(
    save_path = 'QP multi/QP_multi_logs_scenario_3.json',
    bids_dict = bids_dict,
)
print("="*50)



--- 1. QP w/ rep.  (N=100 trials) ---
  Metric                      Mean          SE
  --------------------------------------------
  Revenue per Ad              1.19        0.021
  SocialWelfare               4.57        0.064
  Relevance                   1.86        0.018
  KLDivergence                0.03        0.001
  NumAds                      2.12        0.082

--- 2. QP w/o repl.   (N=100 trials) ---
  Metric                      Mean          SE
  --------------------------------------------
  Revenue per Ad              1.17        0.024
  SocialWelfare               4.06        0.019
  Relevance                   1.82        0.020
  KLDivergence                0.02        0.001
  NumAds                      1.76        0.075

--- 3. QP multi-alloc  (N=100 trials) ---
  Metric                      Mean          SE
  --------------------------------------------
  Revenue per Ad              0.85       0.001
  SocialWelfare               4.94       0.003
  Relevance        

### Seg single & multi

In [29]:
print("\n\n" + "="*50)
print(f"--- 1. Seg w/ rep.  (N={len(json.loads(Path('Seg single & multi/Seg_single_with_replacement_logs_scenario_3.json').read_text()))} trials) ---")
results = evaluate_metrics_from_logs('Seg single & multi/Seg_single_with_replacement_logs_scenario_3.json')
for metric, stats in results.items():
    print(f"  {metric}: {stats['mean']:.2f} ± {stats['se']:.3f}")

print(f"\n--- 2. Seg w/o rep.  (N={len(json.loads(Path('Seg single & multi/Seg_single_without_replacement_logs_scenario_3.json').read_text()))} trials) ---")
results = evaluate_metrics_from_logs('Seg single & multi/Seg_single_without_replacement_logs_scenario_3.json')
for metric, stats in results.items():
    print(f"  {metric}: {stats['mean']:.2f} ± {stats['se']:.3f}")

print(f"\n--- 3. Seg multi-alloc (N={len(json.loads(Path('Seg single & multi/Seg_multi_logs_scenario_3.json').read_text()))} trials) ---")
results = evaluate_metrics_from_logs('Seg single & multi/Seg_multi_logs_scenario_3.json')
for metric, stats in list(results.items())[:-1]:
    print(f"  {metric}: {stats['mean']:.2f} ± {stats['se']:.3f}")
print("="*50)



--- 1. Seg w/ rep.  (N=100 trials) ---
  Revenue per Ad: 1.05 ± 0.045
  Social Welfare: 4.36 ± 0.081
  Relevance: 1.78 ± 0.013
  KL Divergence: 0.26 ± 0.001

--- 2. Seg w/o rep.  (N=100 trials) ---
  Revenue per Ad: 0.97 ± 0.040
  Social Welfare: 3.83 ± 0.035
  Relevance: 1.68 ± 0.010
  KL Divergence: 0.27 ± 0.008

--- 3. Seg multi-alloc (N=100 trials) ---
  Revenue per Ad: 0.78 ± 0.038
  Social Welfare: 3.75 ± 0.033
  Relevance: 1.62 ± 0.010


## Senario 4

### $q_0^{(1)}$

In [30]:
with open('QP single/QP_single_with_replacement_logs_scenario_4.json') as f:
    data = json.load(f)

q0_values = []
for trial in data:
    for r in trial:
        if r['round'] == 1:
            q0_values.append(r['q0'])

q0 = np.array(q0_values)
mean = np.mean(q0)
se = np.std(q0, ddof=1) / np.sqrt(len(q0))
print(f'N = {len(q0)}')
print(f'Mean q0 (round 1) = {mean:.2f}')
print(f'SE = {se:.3f}')

N = 100
Mean q0 (round 1) = 0.78
SE = 0.003


### QP single & multi

In [32]:
bids_dict = {advertiser: 1 for advertiser in ['Velora', 'BookHaven', 'MassMart', 'EspressoEdge', 'SocialHub', 'ColaBubbles', 'FizzyPop', 'SkyTech', 'AeroDynamics', 'MusicStream', 'BrainChips']}

print("\n\n" + "="*50)
print(f"--- 1. QP w/ rep.  (N={len(json.loads(Path('QP single/QP_single_with_replacement_logs_scenario_4.json').read_text()))} trials) ---")
results = compute_metrics_from_file(
    save_path = 'QP single/QP_single_with_replacement_logs_scenario_4.json',
    bids_dict = bids_dict,
)

print(f"\n--- 2. QP w/o repl.   (N={len(json.loads(Path('QP single/QP_single_without_replacement_logs_scenario_4.json').read_text()))} trials) ---")
results = compute_metrics_from_file(
    save_path = 'QP single/QP_single_without_replacement_logs_scenario_4.json',
    bids_dict = bids_dict,
)

print(f"\n--- 3. QP multi-alloc  (N={len(json.loads(Path('QP multi/QP_multi_logs_scenario_4.json').read_text()))} trials) ---")
results = compute_metrics_vcg(
    save_path = 'QP multi/QP_multi_logs_scenario_4.json',
    bids_dict = bids_dict,
)
print("="*50)



--- 1. QP w/ rep.  (N=100 trials) ---
  Metric                      Mean          SE
  --------------------------------------------
  Revenue per Ad              0.52        0.015
  SocialWelfare               1.88        0.013
  Relevance                   2.09        0.016
  KLDivergence                0.00        0.000
  NumAds                      1.91        0.095

--- 2. QP w/o repl.   (N=100 trials) ---
  Metric                      Mean          SE
  --------------------------------------------
  Revenue per Ad              0.50        0.017
  SocialWelfare               1.81        0.006
  Relevance                   2.07        0.019
  KLDivergence                0.00        0.000
  NumAds                      1.74        0.101

--- 3. QP multi-alloc  (N=100 trials) ---
  Metric                      Mean          SE
  --------------------------------------------
  Revenue per Ad              0.23       0.044
  SocialWelfare               3.66       0.063
  Relevance        

### Seg single & multi

In [35]:
print("\n\n" + "="*50)
print(f"--- 1. Seg w/ rep.  (N={len(json.loads(Path('Seg single & multi/Seg_single_with_replacement_logs_scenario_4.json').read_text()))} trials) ---")
results = evaluate_metrics_from_logs('Seg single & multi/Seg_single_with_replacement_logs_scenario_4.json')
for metric, stats in results.items():
    print(f"  {metric}: {stats['mean']:.4f} ± {stats['se']:.4f}")

print(f"\n--- 2. Seg w/o rep.  (N={len(json.loads(Path('Seg single & multi/Seg_single_without_replacement_logs_scenario_4.json').read_text()))} trials) ---")
results = evaluate_metrics_from_logs('Seg single & multi/Seg_single_without_replacement_logs_scenario_4.json')
for metric, stats in results.items():
    print(f"  {metric}: {stats['mean']:.4f} ± {stats['se']:.4f}")

print(f"\n--- 3. Seg multi-alloc (N={len(json.loads(Path('Seg single & multi/Seg_multi_logs_scenario_4.json').read_text()))} trials) ---")
results = evaluate_metrics_from_logs('Seg single & multi/Seg_multi_logs_scenario_4.json')
for metric, stats in list(results.items())[:-1]:
    print(f"  {metric}: {stats['mean']:.4f} ± {stats['se']:.4f}")
print("="*50)



--- 1. Seg w/ rep.  (N=100 trials) ---
  Revenue per Ad: 0.4747 ± 0.0184
  Social Welfare: 1.7207 ± 0.0127
  Relevance: 1.7207 ± 0.0127
  KL Divergence: 0.0000 ± 0.0000

--- 2. Seg w/o rep.  (N=100 trials) ---
  Revenue per Ad: 0.4574 ± 0.0174
  Social Welfare: 1.6886 ± 0.0088
  Relevance: 1.6886 ± 0.0088
  KL Divergence: 0.0000 ± 0.0000

--- 3. Seg multi-alloc (N=100 trials) ---
  Revenue per Ad: 0.4741 ± 0.0152
  Social Welfare: 1.5920 ± 0.0092
  Relevance: 1.5920 ± 0.0092
